# Stage 8A — Investigate model performance

**Goal:** Make an offline prediction log for the held-out November–December requests, join it with their matured 72-hour outcomes, and inspect results by month and borough.

**Why:** A single overall score can hide a problem. Validation recall was `0.878`; test recall was `0.604`. We will measure where that gap appears and save a monitoring report.

This is a **historical simulation**, not a real production log. We use test data only for diagnosis. We do not choose a new threshold or model using these test answers.

In [ ]:
# Cell 2 — Why: verify the exact model and split being monitored.
from google.colab import drive
from pathlib import Path
import hashlib, json, joblib, platform, pandas as pd, numpy as np, sklearn

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
run_id = "20260926T120702352046Z"
run_dir = root / "runs" / run_id
record = json.loads((run_dir / "run.json").read_text())
model_file = run_dir / "model.joblib"
split_dir = root / "splits" / record["split_id"]
split = json.loads((split_dir / "manifest.json").read_text())
if record["run_id"] != run_id or split["split_id"] != record["split_id"]:
    raise ValueError("Wrong run or split")
if hashlib.sha256(model_file.read_bytes()).hexdigest() != record["model_sha256"]:
    raise ValueError("Model file changed")
print("Monitoring model:", run_id, "Split:", record["split_id"])

In [ ]:
# Cell 3 — Why: load only verified data and matching software.
current = {"python": platform.python_version(), "pandas": pd.__version__,
           "numpy": np.__version__, "scikit_learn": sklearn.__version__,
           "joblib": joblib.__version__}
if current != record["versions"]:
    raise RuntimeError("Core software versions differ from model training")
frames = {}
for name, info in split["files"].items():
    file = split_dir / f"{name}.jsonl"
    raw = file.read_bytes()
    if hashlib.sha256(raw).hexdigest() != info["sha256"] or len(raw.splitlines()) != info["rows"]:
        raise ValueError(f"{name} file changed")
    frames[name] = pd.read_json(file, lines=True)
model = joblib.load(model_file)  # Our own hash-verified artifact.
train, valid, test = (frames[name] for name in ["train", "validation", "test"])
print("Verified train/validation/test rows:", [len(train), len(valid), len(test)])

In [ ]:
# Cell 4 — Why: log predictions separately from later outcomes.
prediction_log = test[["unique_key", "created_date", "complaint_type", "borough", "month"]].copy()
prediction_log["model_run_id"] = run_id
prediction_log["probability_late"] = model.predict_proba(test[record["features"]])[:, 1]
prediction_log["predicted_label"] = (
    prediction_log["probability_late"] >= record["decision_threshold"]
).astype(int)
outcomes = test[["unique_key", "label"]].copy()
if prediction_log["unique_key"].duplicated().any() or outcomes["unique_key"].duplicated().any():
    raise ValueError("Duplicate request ID in monitoring data")
print("Offline predictions:", len(prediction_log), "Separate outcomes:", len(outcomes))

In [ ]:
# Cell 5 — Why: join outcomes only after the 72-hour answer is knowable.
dataset = json.loads((root / "datasets" / record["source_dataset_id"] / "manifest.json").read_text())
as_of = pd.Timestamp(dataset["created_at_utc"])
created_local = pd.to_datetime(prediction_log["created_date"], errors="coerce")
created_utc = created_local.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert("UTC")
if created_utc.isna().any() or (created_utc + pd.Timedelta(hours=72) > as_of).any():
    raise ValueError("Some outcomes are not yet eligible for evaluation")
joined = prediction_log.merge(outcomes, on="unique_key", validate="one_to_one")
if len(joined) != len(test) or joined["label"].isna().any():
    raise ValueError("Missing outcome after joining by request ID")
print("Matured outcomes joined:", len(joined), "Snapshot as of:", as_of)

In [ ]:
# Cell 6 — Why: compare November and December separately.
from sklearn.metrics import precision_score, recall_score, f1_score

def group_scores(frame):
    y, p = frame["label"], frame["predicted_label"]
    late, predicted_late = int(y.sum()), int(p.sum())
    return {"rows": len(frame), "late_cases": late,
            "predicted_late": predicted_late,
            "missed_late": int(((y == 1) & (p == 0)).sum()),
            "late_rate": round(float(y.mean()), 3),
            "precision": round(precision_score(y, p), 3) if predicted_late else None,
            "recall": round(recall_score(y, p), 3) if late else None,
            "f1": round(f1_score(y, p), 3) if late + predicted_late else None}

monthly = {str(month): group_scores(group) for month, group in joined.groupby("month")}
print("Monthly monitoring:")
for month, result in monthly.items():
    print(month, result)

In [ ]:
# Cell 7 — Why: find groups hidden by the overall score.
by_borough = {name: group_scores(group) for name, group in joined.groupby("borough")
              if len(group) >= 100}
by_type = {name: group_scores(group) for name, group in joined.groupby("complaint_type")
           if len(group) >= 300}
print("Borough results (at least 100 rows):")
for name, result in sorted(by_borough.items()):
    print(name, result)
print("Lowest recall among types with at least 300 rows and 30 late cases:")
eligible = [(name, result) for name, result in by_type.items()
            if result["late_cases"] >= 30]
for name, result in sorted(eligible, key=lambda pair: pair[1]["recall"])[:8]:
    print(name, result)

In [ ]:
# Cell 8 — Why: check whether the mix of request types changed.
earlier = pd.concat([train, valid], ignore_index=True)
old_share = earlier["complaint_type"].value_counts(normalize=True)
new_share = test["complaint_type"].value_counts(normalize=True)
mix = pd.DataFrame({"earlier_share": old_share, "test_share": new_share}).fillna(0)
mix["change"] = mix["test_share"] - mix["earlier_share"]
biggest_changes = mix.reindex(mix["change"].abs().sort_values(ascending=False).index).head(8)
print("Largest complaint-type share changes:")
print(biggest_changes.round(3))

In [ ]:
# Cell 9 — Why: apply a review trigger and save a report.
from datetime import datetime, timezone

test_result = group_scores(joined)
recall_gap = round(record["candidate_validation"]["recall_late"] - test_result["recall"], 3)
needs_review = recall_gap > 0.10
report_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
report_dir = root / "monitoring" / report_id
report_dir.mkdir(parents=True, exist_ok=False)
print("Validation-to-test recall gap:", recall_gap)
print("Review needed (gap > 0.10):", needs_review)
print("Report ID:", report_id)

In [ ]:
# Cell 10 — Why: preserve separate logs and their provenance.
prediction_file = report_dir / "offline_predictions.jsonl"
outcome_file = report_dir / "matured_outcomes.jsonl"
prediction_log.to_json(prediction_file, orient="records", lines=True, date_format="iso")
outcomes.to_json(outcome_file, orient="records", lines=True)
report = {"report_id": report_id, "kind": "offline historical simulation",
          "model_run_id": run_id, "split_id": record["split_id"],
          "source_dataset_id": record["source_dataset_id"],
          "prediction_sha256": hashlib.sha256(prediction_file.read_bytes()).hexdigest(),
          "outcome_sha256": hashlib.sha256(outcome_file.read_bytes()).hexdigest(),
          "rows_joined": len(joined), "as_of_utc": str(as_of),
          "test_overall": test_result, "monthly": monthly,
          "by_borough_min_100": by_borough, "by_type_min_300": by_type,
          "validation_recall": record["candidate_validation"]["recall_late"],
          "recall_gap": recall_gap, "review_needed": bool(needs_review),
          "review_rule": "validation recall minus test recall > 0.10",
          "lesson_version": "stage8a-v1"}
(report_dir / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("Saved monitoring report:", report_dir / "report.json")

# Stage 8A takeaway and quiz

**What you learned:** We kept prediction logs and outcomes separate, waited until the 72-hour label was knowable, joined by request ID, and inspected monthly and subgroup scores. The recall gap triggers review; it does not prove a cause. These are one-day-per-month samples, so the report is a practice monitor rather than a production performance claim.

**Easy quiz:**

1. Why do we wait at least 72 hours before measuring accuracy for an open request?
2. Why do we join predictions and outcomes by `unique_key`?
3. Why might the overall recall hide a problem for one borough or complaint type?
4. If complaint types changed between periods, does that alone prove why recall fell?

Send me the monthly results, borough results, largest complaint-type changes, review flag, Report ID, and your answers.